# Getting started with Fiatlux

This notebook introduces the basic objects used by Fiatlux:

- `Grid`: sampling of an optical plane
- `Spectrum`: wavelengths and photon flux
- `Source`: source of the optical field
- `CircularAperture`: telescope pupil
- `MFTPropagator`: propagation between pupil and focal planes
- `SerialSystem`: sequence of optical elements

We will simulate the PSF produced by a circular aperture.

In [ ]:
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from fiatlux.core.grid import Grid
from fiatlux.core.spectrum import Band, Spectrum
from fiatlux.core.source import PlaneWave

from fiatlux.optics.elements.mask import CircularAperture
from fiatlux.optics.propagator import MFTPropagator

from fiatlux.system.optical_system import SerialSystem

## Optical parameters

We define a telescope with diameter \(D = 1\) m operating at
\(\lambda = 1.65\,\mu\mathrm{m}\).

The pupil is sampled with 256 × 256 pixels.

In [ ]:
D = 1.0
wavelength = 1.65e-6
focal_length = 10.0

N_pupil = 256

pupil_grid = Grid(
    nx=N_pupil,
    ny=N_pupil,
    dx=D / N_pupil,
    dy=D / N_pupil,
)

pupil_grid

## Define the spectrum

For this first example we use a monochromatic source.

The custom band below has zero bandwidth so the spectrum contains exactly
one wavelength.

In [ ]:
monochromatic_band = Band(
    central_wavelength=wavelength,
    delta_wavelength=0.0,
    f0=368.0,
)

spectrum = Spectrum(
    magnitude=0,
    band=monochromatic_band,
    samples=1,
)

print("Wavelength:", spectrum.wavelengths)
print("Flux:", spectrum.fluxes)

## Define the focal-plane sampling

In [ ]:
N_focal = 256
sampling = 4

dx_focal = focal_length * wavelength / D / sampling

focal_grid = Grid(
    nx=N_focal,
    ny=N_focal,
    dx=dx_focal,
    dy=dx_focal,
)

## Create the source and optical system

In [ ]:
source = PlaneWave(spectrum=spectrum)

aperture = CircularAperture(
    grid=pupil_grid,
    radius=D / 2,
)

propagator = MFTPropagator(
    focal_length=focal_length,
    output_grid=focal_grid,
)

system = SerialSystem(
    elements=[
        aperture,
        propagator,
    ]
)

print(system)

## Propagate the field

In [ ]:
result = system.run(source=source)

## Inspect the pupil

In [ ]:
pupil_field = result.field_at(aperture)

plt.figure(figsize=(5, 5))
plt.imshow(
    pupil_field.intensity()[0].cpu(),
    extent=[
        float(pupil_grid.x.min()),
        float(pupil_grid.x.max()),
        float(pupil_grid.y.min()),
        float(pupil_grid.y.max()),
    ],
    origin="lower",
)

plt.xlabel("x [m]")
plt.ylabel("y [m]")
plt.title("Entrance pupil")
plt.colorbar()
plt.show()

## Display the PSF

In [ ]:
focal_field = result.field_at(propagator)

psf = focal_field.intensity().sum(dim=0)
psf = psf / psf.max()

extent_lambda_D = [
    float(focal_grid.x.min() / (focal_length * wavelength / D)),
    float(focal_grid.x.max() / (focal_length * wavelength / D)),
    float(focal_grid.y.min() / (focal_length * wavelength / D)),
    float(focal_grid.y.max() / (focal_length * wavelength / D)),
]

plt.figure(figsize=(6, 5))
plt.imshow(
    psf.cpu(),
    extent=extent_lambda_D,
    origin="lower",
    norm=LogNorm(vmin=1e-6, vmax=1),
)

plt.xlabel(r"$x\ [\lambda/D]$")
plt.ylabel(r"$y\ [\lambda/D]$")
plt.title("Circular aperture PSF")
plt.colorbar(label="Normalized intensity")
plt.xlim(-10, 10)
plt.ylim(-10, 10)
plt.show()